# Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay

## Paper Citation
- Title: Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay
- Authors: Chorok Lee
- Published: 2025-12-11
- ArXiv: [https://arxiv.org/abs/2512.11913](https://arxiv.org/abs/2512.11913)

## Strategy Description
This notebook implements a quantitative trading strategy based on the paper "Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay" by Chorok Lee. The strategy focuses on trading based on factor alpha decay, specifically using momentum as a primary signal. The decay function used is hyperbolic: alpha(t) = K/(1+lambda*t). The strategy aims to capture the decay in factor alpha over time, particularly for momentum, and adjust positions accordingly.


In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
START_DATE = '2010-01-01'
END_DATE = '2023-12-31'
MOMENTUM_PERIOD = 12
DECAY_RATE = 0.05
POSITION_SIZE = 0.02

# Hypothesis
# The strategy hypothesizes that momentum factors exhibit hyperbolic decay over time.
# By modeling this decay, the strategy aims to capture alpha before it diminishes.


## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE, group_by='ticker')

# Compute momentum
momentum = data['Adj Close'].pct_change(MOMENTUM_PERIOD)

# Cross-sectional normalization
normalized_momentum = momentum.sub(momentum.mean(axis=1), axis=0).div(momentum.std(axis=1), axis=0)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
signals = normalized_momentum.shift(1)

# Position sizing
positions = signals.mul(POSITION_SIZE)

# Portfolio construction
portfolio = positions.mul(data['Adj Close'].shift(1))

## Phase 4 — Vectorized Backtest

In [ ]:
# Daily returns
daily_returns = portfolio.pct_change()

# Cumulative returns
cumulative_returns = (1 + daily_returns).cumprod() - 1

## Phase 5 — Performance Metrics

In [ ]:
import scipy.stats as stats

# Performance metrics
sharpe_ratio = np.mean(daily_returns) / np.std(daily_returns)
sortino_ratio = np.mean(daily_returns) / np.std(daily_returns[daily_returns < 0])
max_drawdown = cumulative_returns.cummax() - cumulative_returns
calmar_ratio = sharpe_ratio / max_drawdown.max()

print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown.max():.2f}')

import matplotlib.pyplot as plt

# Plot equity curve
plt.figure(figsize=(10, 5))
plt.plot(cumulative_returns.iloc[:, 0], label='Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.title('Equity Curve')
plt.legend()
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor_daily_pnl(data, positions):
    daily_pnl = positions.mul(data['Adj Close'].pct_change())
    print(f'Daily P&L: {daily_pnl.sum().sum():.2f}')
    print(f'Current Positions: {positions.iloc[-1]}')

# Example usage
monitor_daily_pnl(data, positions)